# 05 — Transformer Decoder, Masking, and Training

    **Companion chapter:** `05-transformer-decoder-and-training.md`

    ## Learning goals

    - Create shifted decoder inputs.
- Build and inspect a causal mask.
- Implement cross-attention.
- Calculate token-level cross-entropy.
- Assemble a decoder layer and vocabulary projection.

    ## How to use this notebook

    Run the cells from top to bottom. Read the comments, change small values, and
    rerun the cell. Every notebook ends with practice prompts that can become
    GitHub issues, exercises, or discussion questions.

In [1]:
from __future__ import annotations

import math
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## 1. Shift the target sequence

During teacher-forced training, decoder inputs start with `BOS`, while the
expected targets end with `EOS`.

In [2]:
BOS, EOS, PAD = 1, 2, 0
target = torch.tensor([[5, 6, 7, 8, EOS]])  # I read the book EOS

decoder_input = torch.cat(
    [torch.full((target.shape[0], 1), BOS), target[:, :-1]],
    dim=1,
)

print("Decoder input:", decoder_input.tolist())
print("Training target:", target.tolist())

Decoder input: [[1, 5, 6, 7, 8]]
Training target: [[5, 6, 7, 8, 2]]


## 2. Construct a causal mask

A True value below means "blocked" for PyTorch's attention modules.

In [3]:
def causal_mask(length: int, device: torch.device | str = "cpu") -> torch.Tensor:
    return torch.triu(
        torch.ones(length, length, dtype=torch.bool, device=device),
        diagonal=1,
    )


mask = causal_mask(5)
print(mask.int())

plt.imshow(mask.int().numpy())
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.title("Causal mask: 1 means future position is blocked")
plt.show()

tensor([[0, 1, 1, 1, 1],
        [0, 0, 1, 1, 1],
        [0, 0, 0, 1, 1],
        [0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0]], dtype=torch.int32)


/tmp/ipykernel_791/1360468611.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Observe masked self-attention probabilities

Future scores are filled with `-∞` before softmax.

In [4]:
x = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
    ]
)

scores = x @ x.T / math.sqrt(x.shape[-1])
additive_mask = torch.zeros_like(scores)
additive_mask[causal_mask(len(x))] = float("-inf")

masked_weights = torch.softmax(scores + additive_mask, dim=-1)

print("Masked attention weights:")
print(masked_weights.round(decimals=3))
assert torch.allclose(masked_weights.triu(diagonal=1), torch.zeros_like(masked_weights))

Masked attention weights:
tensor([[1.0000, 0.0000, 0.0000],
        [0.3300, 0.6700, 0.0000],
        [0.2480, 0.2480, 0.5030]])


## 4. Cross-attention

Decoder representations provide queries. Encoder outputs provide keys and values.
Source and target lengths may therefore differ.

In [5]:
class SingleHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, d_k: int):
        super().__init__()
        self.q = nn.Linear(d_model, d_k)
        self.k = nn.Linear(d_model, d_k)
        self.v = nn.Linear(d_model, d_k)
        self.output = nn.Linear(d_k, d_model)

    def forward(
        self,
        decoder_states: torch.Tensor,
        encoder_states: torch.Tensor,
    ):
        q = self.q(decoder_states)
        k = self.k(encoder_states)
        v = self.v(encoder_states)

        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        weights = torch.softmax(scores, dim=-1)
        context = weights @ v
        return self.output(context), weights


decoder_states = torch.randn(2, 5, 8)  # batch, target_len, d_model
encoder_states = torch.randn(2, 7, 8)  # batch, source_len, d_model

cross_attention = SingleHeadCrossAttention(d_model=8, d_k=4)
context, cross_weights = cross_attention(decoder_states, encoder_states)

print("Context:", tuple(context.shape))
print("Cross-attention weights:", tuple(cross_weights.shape))

Context: (2, 5, 8)
Cross-attention weights: (2, 5, 7)


## 5. Token-level cross-entropy

Cross-entropy strongly penalizes a model that assigns a very small probability
to the correct token.

In [6]:
probabilities = torch.tensor([0.7, 0.01])
losses = -torch.log(probabilities)

pd.DataFrame(
    {
        "correct-token probability": probabilities.numpy(),
        "negative log-likelihood": losses.numpy(),
    }
)

,correct-token probability,negative log-likelihood
0,0.70,0.356675
1,0.01,4.605170


In [7]:
vocab_size = 10
logits = torch.randn(2, 5, vocab_size)
gold_targets = torch.randint(0, vocab_size, (2, 5))
gold_targets[1, -2:] = PAD

loss_function = nn.CrossEntropyLoss(ignore_index=PAD)
loss = loss_function(logits.reshape(-1, vocab_size), gold_targets.reshape(-1))

print("Cross-entropy:", round(loss.item(), 4))

Cross-entropy: 2.8714


## 6. A small Transformer decoder block

This block combines masked self-attention, cross-attention, and an FFN.

In [8]:
class TeachingDecoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int):
        super().__init__()
        self.self_attention = nn.MultiheadAttention(
            d_model, num_heads, batch_first=True
        )
        self.cross_attention = nn.MultiheadAttention(
            d_model, num_heads, batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(
        self,
        target_states: torch.Tensor,
        encoder_states: torch.Tensor,
    ):
        length = target_states.shape[1]
        mask = causal_mask(length, target_states.device)

        self_output, self_weights = self.self_attention(
            target_states,
            target_states,
            target_states,
            attn_mask=mask,
            need_weights=True,
            average_attn_weights=False,
        )
        x = self.norm1(target_states + self_output)

        cross_output, cross_weights = self.cross_attention(
            x,
            encoder_states,
            encoder_states,
            need_weights=True,
            average_attn_weights=False,
        )
        x = self.norm2(x + cross_output)
        y = self.norm3(x + self.ffn(x))
        return y, self_weights, cross_weights


decoder_layer = TeachingDecoderLayer(d_model=8, num_heads=2, d_ff=32)
decoder_output, self_weights, cross_weights = decoder_layer(
    decoder_states, encoder_states
)

print("Decoder output:", tuple(decoder_output.shape))
print("Masked self-attention:", tuple(self_weights.shape))
print("Cross-attention:", tuple(cross_weights.shape))

Decoder output: (2, 5, 8)
Masked self-attention: (2, 2, 5, 5)
Cross-attention: (2, 2, 5, 7)


## 7. Vocabulary projection

The final decoder vectors are mapped to one logit per vocabulary item.

In [9]:
output_projection = nn.Linear(8, vocab_size)
vocabulary_logits = output_projection(decoder_output)
next_token_probabilities = torch.softmax(vocabulary_logits, dim=-1)

print("Vocabulary logits:", tuple(vocabulary_logits.shape))
print("One distribution sums to:", next_token_probabilities[0, 0].sum().item())
print("Greedy token IDs:", vocabulary_logits.argmax(dim=-1)[0].tolist())

Vocabulary logits: (2, 5, 10)
One distribution sums to: 1.0
Greedy token IDs: [8, 3, 4, 3, 7]


## Practice

1. Verify that the first target position attends only to itself.
2. Remove the causal mask and explain the information leak.
3. Change source length from 7 to 10 and inspect cross-attention shape.
4. Add target padding and pass a target key-padding mask.
5. Implement a one-step greedy decoding loop with a toy vocabulary.